# YT Agent — Kaggle GPU worker

Triggered by the `kaggle-dispatch.yml` GitHub Actions workflow whenever the
Vercel `/api/maintenance/needs-worker` route reports that a render is queued
and no GPU worker is alive. Boots an ephemeral FastAPI backend on a free
Kaggle T4 / P100, registers itself in Firestore, claims the queued job, runs
the pipeline, then self-terminates after 10 min of idle to preserve the
30 GPU hr/week budget.

**One-time setup**: see `kaggle/README.md`. You need to add
`GOOGLE_APPLICATION_CREDENTIALS_JSON` (and optionally R2 / SFTP) to the
Kaggle Add-ons → Secrets panel for THIS notebook.

In [ ]:
# 1) System deps + repo clone
import os, subprocess
subprocess.check_call(['apt-get', '-qq', 'update'])
# fonts-noto{,-cjk,-color-emoji} + fonts-dejavu-core — needed for
# burned-in captions to render non-Latin scripts (Devanagari, Bengali,
# CJK, Arabic, Thai) instead of tofu boxes. DejaVu Sans is the primary
# font (bundled default) with libass falling back to Noto CJK for CJK.
subprocess.check_call(['apt-get', '-qq', 'install', '-y',
                       'ffmpeg', 'fonts-dejavu-core',
                       'fonts-noto', 'fonts-noto-cjk',
                       'fonts-noto-color-emoji'])
subprocess.run(['fc-cache', '-f'], check=False)

REPO_URL = 'https://github.com/Ahsan3301/yt_agent.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/yt_agent'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
os.chdir(REPO_DIR)
print('repo at:', os.getcwd())
print('commit:', subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 2) Python deps - base + GPU extras + browser agent + Kokoro TTS + local SDXL
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
# Kaggle's base image already includes a CUDA torch; install the GPU
# extras NOT pinned to a specific torch wheel (skip torch line to
# avoid clobbering Kaggle's preinstalled CUDA torch).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'decord==0.6.0', 'av==12.3.0'])
# Kokoro TTS + espeak-ng apt binary. Kaggle's base image ships vanilla
# `phonemizer` — its EspeakWrapper doesn't have set_data_path(), which
# is what misaki (Kokoro's phoneme dep) calls at import time. If we
# leave it installed, pip's install of phonemizer-fork lands ALONGSIDE
# and the older one wins on `import phonemizer`. Uninstall it FIRST,
# then install the fork with --force-reinstall so the correct wheel
# definitely lands.
subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'espeak-ng'])
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'phonemizer'], check=False)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'phonemizer-fork>=3.3.2'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'espeakng-loader>=0.2',
    'soundfile>=0.12',
    'kokoro>=0.9.4',
    'misaki>=0.9.4',
])
# hf_transfer — Rust-based parallel HTTPS downloader for HuggingFace.
# 3-5x faster than the default requests-based path. Cuts the SDXL first-load
# download (~7 GB) from ~4 min to ~1 min. Also speeds up Kokoro's ~330 MB
# model fetch. The env var must be set BEFORE any diffusers/huggingface_hub
# import so the accelerated backend gets selected.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'hf_transfer'])
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# Local SDXL image gen — used by the local_sdxl provider in
# shotfinder.py. Uses Kaggle's preinstalled CUDA torch.
# Install diffusers/transformers/accelerate WITHOUT letting them
# upgrade torch. Kaggle's base image ships a torch wheel compiled
# with sm_60 support for the P100; the latest torch on PyPI dropped
# sm_60, so an unpinned diffusers install would silently pull a
# newer torch that hits `cudaErrorNoKernelImageForDevice` at
# pipe.to('cuda'). --no-deps + explicit deps we know we need keeps
# the working torch untouched.
# transformers, accelerate, safetensors are needed by diffusers at
# import + generate time; huggingface_hub is the download layer.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    'diffusers>=0.30',
])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40',
    'accelerate>=0.30',
    'safetensors>=0.4',
    'huggingface_hub>=0.23',
    'peft>=0.10',
])
# Force-reinstall edge-tts >=7 to dodge the Microsoft auth-endpoint
# breakage in older wheels.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'edge-tts>=7.0.0'])
# Playwright + beautifulsoup for the NIM-driven research agent.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-browser.txt'])
# --with-deps: apt-installs the shared libs Chromium needs (libnss3, libatk*, libgbm, etc.).
# Without them the download succeeds but chromium.launch() raises 'error while loading shared libraries'
# and the research_agent silently falls back to text-only research.
subprocess.check_call([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'])
import torch
print('torch:', torch.__version__, 'cuda available:', torch.cuda.is_available())
try:
    from kokoro import KPipeline
    print('kokoro: ready')
except Exception as e:
    print('kokoro: NOT ready ->', e)
try:
    import edge_tts
    print('edge-tts:', getattr(edge_tts, '__version__', 'installed'))
except Exception as e:
    print('edge-tts: NOT ready ->', e)
try:
    from diffusers import AutoPipelineForText2Image  # noqa: F401
    print('diffusers: ready (local_sdxl provider available)')
except Exception as e:
    print('diffusers: NOT ready ->', e)
print('playwright + chromium ready')

In [ ]:
# 3) BOOTSTRAP secrets.
#
# Two sources, in priority order:
#
#   (1) A private Kaggle Dataset attached via kernel-metadata.json. This
#       is the production path — survives `kaggle kernels push` cycles
#       (each push creates a new kernel version, but the dataset stays
#       attached). Create one private Dataset named e.g.
#       "yt-agent-secrets" with ONE file:
#
#           secrets.env
#               COOLIFY_BASE_URL=https://yt-agent.thyker.online
#               PB_URL=https://yt-agent.thyker.online/pb
#               POCKETBASE_ADMIN_EMAIL=admin@yt-agent.thyker.online
#               POCKETBASE_ADMIN_PASSWORD=your-password
#               RENDER_TRIGGER_KEY=...
#               STORAGE_PROVIDERS_ENC_KEY=...
#
#       Then reference it in kernel-metadata.json:
#           "dataset_sources": ["<your-username>/yt-agent-secrets"]
#
#   (2) Kaggle's Secrets panel (Add-ons → Secrets). Useful for
#       interactive testing. Same key names as the .env file.
#
# Either source works — the notebook reads whichever it finds first.
import os, glob

REQUIRED = [
    'COOLIFY_BASE_URL',
    'PB_URL',
    'POCKETBASE_ADMIN_EMAIL',
    'POCKETBASE_ADMIN_PASSWORD',
    'RENDER_TRIGGER_KEY',
    'STORAGE_PROVIDERS_ENC_KEY',
]
OPTIONAL = [
    # Legacy Vercel + Firestore path — keep working for users who
    # haven\'t migrated to Coolify yet.
    'GOOGLE_APPLICATION_CREDENTIALS_JSON',
    'GOOGLE_APPLICATION_CREDENTIALS_JSON_B64',
    'INSTANCE_LABEL', 'INSTANCE_TIER',
]
ALL_KEYS = REQUIRED + OPTIONAL


def _load_from_dataset_envfile():
    """Look for an attached Dataset secrets file. Kaggle mounts datasets
    under /kaggle/input/<dataset-name>/. Search for the most common
    file names."""
    candidates = []
    for root in glob.glob('/kaggle/input/*/'):
        for name in ('secrets.env', '.env', 'env.txt', 'secrets.txt'):
            p = os.path.join(root, name)
            if os.path.exists(p):
                candidates.append(p)
    if not candidates:
        return False
    found = False
    for path in candidates:
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                k, v = line.split('=', 1)
                k = k.strip()
                v = v.strip().strip('"').strip("'")
                if k in ALL_KEYS and v:
                    os.environ.setdefault(k, v)
                    found = True
        print(f'  loaded secrets from {path}')
    return found


def _load_from_kaggle_secrets():
    """Read from the Kaggle Secrets panel. Fails silently when not
    available (e.g. running in a Colab clone of the notebook)."""
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        loaded = 0
        for k in ALL_KEYS:
            try:
                v = secrets.get_secret(k)
            except Exception:
                continue
            if v and not os.environ.get(k):
                os.environ[k] = v
                loaded += 1
        if loaded:
            print(f'  loaded {loaded} secrets from Kaggle Secrets panel')
        return loaded > 0
    except Exception as e:
        print(f'  Kaggle Secrets unavailable ({e!r}); using Dataset only')
        return False


# Try both sources — Dataset first, then Kaggle Secrets fills any gaps.
got_dataset = _load_from_dataset_envfile()
got_secrets = _load_from_kaggle_secrets()
if not (got_dataset or got_secrets):
    raise SystemExit(
        'No secrets source found. Either attach the yt-agent-secrets '
        'Dataset (via kernel-metadata.json dataset_sources) OR add the '
        'required keys to the Kaggle Secrets panel.'
    )

missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise SystemExit('MISSING REQUIRED SECRETS: ' + ', '.join(missing))

# Defaults for the new outbound-poll mode.
os.environ.setdefault('DB_BACKEND', 'pocketbase')
os.environ.setdefault('STORAGE_BACKEND', 'registry')
os.environ.setdefault('WORKER_MODE', 'outbound_poll')
os.environ.setdefault('INSTANCE_TIER', 'gpu')
os.environ.setdefault('INSTANCE_LABEL', 'kaggle-gpu')
# Idle auto-shutdown ~25 min — preserves the 30 GPU-hr/week budget.
os.environ.setdefault('KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS', '1500')

print('Kaggle bootstrap OK')
print('  Dashboard:', os.environ.get('COOLIFY_BASE_URL'))
print('  Pocketbase:', os.environ.get('PB_URL'))
print('  Mode:', os.environ.get('WORKER_MODE'), '| tier:', os.environ.get('INSTANCE_TIER'))


In [ ]:
# 4) (Skipped in outbound-poll mode.) Legacy cloudflared tunnel cell.
import os, subprocess, time, re

if os.environ.get('WORKER_MODE', 'tunnel').lower() != 'tunnel':
    print('outbound-poll mode — no cloudflared tunnel needed.')
else:
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.check_call([
            'wget', '-q',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-O', '/usr/local/bin/cloudflared',
        ])
        subprocess.check_call(['chmod', '+x', '/usr/local/bin/cloudflared'])
    tunnel_log = '/tmp/cloudflared.log'
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000',
         '--logfile', tunnel_log, '--loglevel', 'info'],
        stdout=open(tunnel_log + '.stdout', 'w'),
        stderr=subprocess.STDOUT,
    )
    url = None
    for _ in range(40):
        time.sleep(0.5)
        if not os.path.exists(tunnel_log):
            continue
        with open(tunnel_log) as f:
            m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', f.read())
            if m:
                url = m.group(0)
                break
    if not url:
        raise RuntimeError('cloudflared did not produce a URL')
    os.environ['PUBLIC_BACKEND_URL'] = url
    print('Public backend URL:', url)


In [ ]:
# 4.5) Pre-warm SDXL model download + Kokoro touch (boot time).
#
# Without this, the FIRST job after a fresh Kaggle boot pays the full
# ~4 min SDXL download + ~1 min materialize on cuda:0 AND ~1 min on
# cuda:1 (bg warm helps but the first job's step 2 is often shorter
# than the warm). Pre-downloading the weights into the HF cache here
# makes subsequent .from_pretrained() calls skip the network step.
# hf_transfer + parallel across both cards keeps this to ~90-120s.
import os, sys, time, subprocess
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

# Boot pre-warm is a hint — never block the server if any of it fails.
try:
    t0 = time.time()
    print('preboot: snapshotting SDXL weights into HF cache...')
    from huggingface_hub import snapshot_download
    _sdxl_model = os.environ.get('LOCAL_SDXL_MODEL', 'stabilityai/sdxl-turbo')
    snapshot_download(
        _sdxl_model,
        allow_patterns=['*.json', '*.txt', '*.safetensors'],
    )
    print(f'preboot: SDXL weights cached in {time.time()-t0:.1f}s')
except Exception as e:
    print(f'preboot: SDXL cache warm skipped ({e!r})')

# Touch Kokoro so the first render doesn't pay the phonemizer +
# espeak-ng + model-file load. This does NOT keep the pipeline
# in memory (that process ends when this cell finishes) — it just
# validates the deps are usable and pulls the ~330MB model file
# into the HF cache. The uvicorn process will re-instantiate but
# from the warm on-disk cache.
try:
    t1 = time.time()
    from kokoro import KPipeline
    _kp = KPipeline(lang_code='a')
    # One tiny generate to force the underlying model to load fully.
    for _ in _kp('warmup', voice='am_michael', speed=1.0):
        break
    del _kp
    print(f'preboot: Kokoro warm in {time.time()-t1:.1f}s')
except Exception as e:
    print(f'preboot: Kokoro warm skipped ({e!r})')

# nvidia-smi visibility check — if this only shows one GPU on a T4x2
# accelerator, the user forgot to click "T4 x2" in the accelerator
# picker after their last kernel push. Log so it's obvious.
try:
    out = subprocess.check_output(['nvidia-smi', '-L'], timeout=5).decode().strip()
    print('preboot: nvidia-smi -L:\n' + out)
    if out.count('GPU ') < 2:
        print('preboot: WARNING — only 1 GPU visible. To use T4x2, set '
              'Notebook Settings → Accelerator → GPU T4 x2 → Save Version.')
except Exception as e:
    print(f'preboot: nvidia-smi -L failed ({e!r})')

In [ ]:
# 5) Boot the backend (BLOCKING). idle_watchdog will os._exit(0) after
#    KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS once the queue is empty.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'uvicorn', 'backend.server:app',
    '--host', '0.0.0.0', '--port', '8000',
    '--log-level', 'info',
])